# Data Cleaning – Egypt Energy Datasets


## 1. Setup

In [ ]:
import pandas as pd
from pathlib import Path

# Folder that contains the CSV files ("." = same folder as this notebook)
DATA_DIR = Path(".")
# Folder where the cleaned files will be saved
OUT_DIR = Path("cleaned")
OUT_DIR.mkdir(exist_ok=True)

START_YEAR = 2015   # keep rows with Year >= 2015

FILES = [
    "1_solar.csv",
    "2_wind.csv",
    "3_electricity_consumption.csv",
    "4_fuel_burned.csv",
    "5_imports.csv",
    "6_gas.csv",
]

## 2. Load the raw files

In [ ]:
raw = {f: pd.read_csv(DATA_DIR / f) for f in FILES}

for name, df in raw.items():
    print(f"{name:35s} shape={df.shape}  years={df['Year'].min()}-{df['Year'].max()}")

## 3. Cleaning function


In [ ]:
def clean(df: pd.DataFrame, start_year: int = START_YEAR) -> pd.DataFrame:
    df = df.copy()

    # 1. tidy column names
    df.columns = df.columns.str.strip()

    # 2. make sure Year is a valid integer
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df = df.dropna(subset=["Year"])
    df["Year"] = df["Year"].astype(int)

    # 3. all other columns must be numeric
    value_cols = [c for c in df.columns if c != "Year"]
    df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")

    # 4. remove duplicates
    df = df.drop_duplicates()
    df = df.drop_duplicates(subset="Year", keep="last")

    # 5. remove everything before 2015
    df = df[df["Year"] >= start_year]

    # 6. sort and reset index
    return df.sort_values("Year").reset_index(drop=True)

## 4. Apply cleaning

In [ ]:
cleaned = {f: clean(df) for f, df in raw.items()}

for name in FILES:
    before, after = raw[name], cleaned[name]
    print(f"{name:35s} rows: {len(before):3d} -> {len(after):3d}   "
          f"years: {after['Year'].min()}-{after['Year'].max()}")

## 5. Check results

In [ ]:
# No year before 2015 should remain
assert all(df["Year"].min() >= START_YEAR for df in cleaned.values())
print("OK: no data before", START_YEAR)

# Missing values that remain after 2015
for name, df in cleaned.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"\n{name}")
    print(missing.to_string() if not missing.empty else "  no missing values")

## 6. Preview

In [ ]:
for name, df in cleaned.items():
    print("=" * 80)
    print(name)
    display(df.head(3))

## 7. Save cleaned files

In [ ]:
for name, df in cleaned.items():
    out = OUT_DIR / name.replace(".csv", "_cleaned.csv")
    df.to_csv(out, index=False)
    print("saved ->", out)